In [7]:
#CO3_AT1.1
import nltk
from collections import Counter
from nltk.util import ngrams

nltk.download("punkt")

corpus = [
    "Artificial intelligence is changing the world.",
    "Artificial intelligence is transforming industries.",
    "Artificial intelligence improves business.",
    "Machine learning is a part of artificial intelligence.",
    "Machine learning can solve complex problems.",
    "Machine learning improves prediction.",
    "Deep learning is used in artificial intelligence."
]

text = " ".join(corpus).lower()
tokens = nltk.word_tokenize(text)

unigram = Counter(tokens)
bigram = Counter(ngrams(tokens, 2))
trigram = Counter(ngrams(tokens, 3))

print("UNIGRAM COUNTS")
print(unigram)

print("\nBIGRAM COUNTS")
print(bigram)

print("\nTRIGRAM COUNTS")
print(trigram)

def predict(sentence, n):
    words = nltk.word_tokenize(sentence.lower())

    if n == 1:
        candidates = {
            word: count / len(tokens)
            for word, count in unigram.items()
        }

    elif n == 2:
        last = words[-1]
        total = unigram[last]

        candidates = {
            b[1]: count / total
            for b, count in bigram.items()
            if b[0] == last
        }

    else:
        last2 = tuple(words[-2:])
        total = bigram[last2]

        candidates = {
            t[2]: count / total
            for t, count in trigram.items()
            if t[:2] == last2
        }

    return sorted(
        candidates.items(),
        key=lambda x: x[1],
        reverse=True
    )[:5]

print("\nTOP-5 PREDICTIONS")

for n in [1, 2, 3]:
    print(f"\nN = {n}")
    try:
        print(predict("Artificial intelligence is", n))
    except:
        print("Unseen N-gram")

print("\nUNSEEN N-GRAM TEST")

try:
    print(predict("Artificial intelligence xyz", 3))
except:
    print("Zero probability: N-gram not found")

test_sentences = [
    "Artificial intelligence is",
    "Machine learning is",
    "Deep learning is"
]

correct = 0
total = len(test_sentences)

for sentence in test_sentences:
    result = predict(sentence, 3)

    if result:
        correct += 1

accuracy = correct / total * 100

print("\nPREDICTION ACCURACY")
print(f"{accuracy:.2f}%")

print("\nLIMITATIONS")
print("1. Unseen N-grams receive zero probability.")
print("2. Large corpus is required.")
print("3. Long-distance dependencies are not captured.")
print("4. Vocabulary size can become very large.")

UNIGRAM COUNTS
Counter({'.': 7, 'artificial': 5, 'intelligence': 5, 'is': 4, 'learning': 4, 'machine': 3, 'improves': 2, 'changing': 1, 'the': 1, 'world': 1, 'transforming': 1, 'industries': 1, 'business': 1, 'a': 1, 'part': 1, 'of': 1, 'can': 1, 'solve': 1, 'complex': 1, 'problems': 1, 'prediction': 1, 'deep': 1, 'used': 1, 'in': 1})

BIGRAM COUNTS
Counter({('artificial', 'intelligence'): 5, ('.', 'machine'): 3, ('machine', 'learning'): 3, ('intelligence', 'is'): 2, ('.', 'artificial'): 2, ('learning', 'is'): 2, ('intelligence', '.'): 2, ('is', 'changing'): 1, ('changing', 'the'): 1, ('the', 'world'): 1, ('world', '.'): 1, ('is', 'transforming'): 1, ('transforming', 'industries'): 1, ('industries', '.'): 1, ('intelligence', 'improves'): 1, ('improves', 'business'): 1, ('business', '.'): 1, ('is', 'a'): 1, ('a', 'part'): 1, ('part', 'of'): 1, ('of', 'artificial'): 1, ('learning', 'can'): 1, ('can', 'solve'): 1, ('solve', 'complex'): 1, ('complex', 'problems'): 1, ('problems', '.'): 1, 

[nltk_data] Downloading package punkt to /root/nltk_data...
[nltk_data]   Package punkt is already up-to-date!


In [8]:
#CO3_AT1.2
import nltk
from collections import Counter
from nltk.util import ngrams

nltk.download("punkt")

corpus = [
    "machine learning can solve problems",
    "machine learning can improve business",
    "machine learning improves prediction",
    "artificial intelligence can solve problems",
    "artificial intelligence improves healthcare",
    "deep learning can analyze data"
]

tokens = nltk.word_tokenize(" ".join(corpus).lower())

uni = Counter(tokens)
bi = Counter(ngrams(tokens, 2))
tri = Counter(ngrams(tokens, 3))

total = len(tokens)

def unigram_probability(word):
    return uni[word] / total

def bigram_probability(w1, w2):
    if bi[(w1, w2)] == 0:
        return 0
    return bi[(w1, w2)] / uni[w1]

def trigram_probability(w1, w2, w3):
    if tri[(w1, w2, w3)] == 0:
        return 0
    return tri[(w1, w2, w3)] / bi[(w1, w2)]

def unsmoothed(sentence):
    words = nltk.word_tokenize(sentence.lower())
    context = tuple(words[-2:])

    result = {}

    for word in uni:
        p = trigram_probability(context[0], context[1], word)

        if p > 0:
            result[word] = p

    return sorted(result.items(), key=lambda x: x[1], reverse=True)[:5]

def backoff(sentence):
    words = nltk.word_tokenize(sentence.lower())
    w1, w2 = words[-2:]

    result = {}

    for word in uni:
        p3 = trigram_probability(w1, w2, word)

        if p3 > 0:
            result[word] = p3
        else:
            p2 = bigram_probability(w2, word)

            if p2 > 0:
                result[word] = p2
            else:
                result[word] = unigram_probability(word)

    return sorted(result.items(), key=lambda x: x[1], reverse=True)[:5]

lambda1 = 0.5
lambda2 = 0.3
lambda3 = 0.2

def interpolation(sentence):
    words = nltk.word_tokenize(sentence.lower())
    w1, w2 = words[-2:]

    result = {}

    for word in uni:
        p3 = trigram_probability(w1, w2, word)
        p2 = bigram_probability(w2, word)
        p1 = unigram_probability(word)

        probability = (
            lambda1 * p3 +
            lambda2 * p2 +
            lambda3 * p1
        )

        result[word] = probability

    return sorted(result.items(), key=lambda x: x[1], reverse=True)[:5]

sentence = "Machine learning can"

print("=" * 50)
print("BACKOFF AND INTERPOLATION")
print("=" * 50)

print("\nSentence:", sentence)

print("\nUNSMOOTHED N-GRAM")
print(unsmoothed(sentence))

print("\nBACKOFF MODEL")
print(backoff(sentence))

print("\nDELETED INTERPOLATION")
print(interpolation(sentence))

print("\nINTERPOLATION WEIGHTS")
print("Lambda Trigram =", lambda1)
print("Lambda Bigram  =", lambda2)
print("Lambda Unigram =", lambda3)

print("\nCOMPARISON")
print("Unsmoothed : unseen N-grams get probability 0")
print("Backoff    : uses lower-order N-grams")
print("Interpolation: combines all N-gram probabilities")
print("Backoff and interpolation improve prediction coverage.")

BACKOFF AND INTERPOLATION

Sentence: Machine learning can

UNSMOOTHED N-GRAM
[('solve', 0.3333333333333333), ('improve', 0.3333333333333333), ('analyze', 0.3333333333333333)]

BACKOFF MODEL
[('solve', 0.3333333333333333), ('improve', 0.3333333333333333), ('analyze', 0.3333333333333333), ('learning', 0.14285714285714285), ('can', 0.14285714285714285)]

DELETED INTERPOLATION
[('solve', 0.33095238095238094), ('improve', 0.24880952380952379), ('analyze', 0.24880952380952379), ('learning', 0.02857142857142857), ('can', 0.02857142857142857)]

INTERPOLATION WEIGHTS
Lambda Trigram = 0.5
Lambda Bigram  = 0.3
Lambda Unigram = 0.2

COMPARISON
Unsmoothed : unseen N-grams get probability 0
Backoff    : uses lower-order N-grams
Interpolation: combines all N-gram probabilities
Backoff and interpolation improve prediction coverage.


[nltk_data] Downloading package punkt to /root/nltk_data...
[nltk_data]   Package punkt is already up-to-date!


In [11]:
#CO3_AT1.3
import nltk
import math
from collections import Counter
from nltk.util import ngrams

nltk.download("punkt")

train = [
    "the cat sits on the mat",
    "the cat eats food",
    "the dog sits on the mat",
    "the dog eats food",
    "the boy sits on the chair",
    "the girl reads a book"
]

test = [
    "the cat sits on the mat",
    "the quantum processor redesigned the"
]

tokens = nltk.word_tokenize(" ".join(train).lower())

unigram = Counter(tokens)
bigram = Counter(ngrams(tokens, 2))

total = len(tokens)

def unigram_prob(word):
    if unigram[word] == 0:
        return 0.000001
    return unigram[word] / total

def bigram_prob(w1, w2):
    if bigram[(w1, w2)] == 0:
        return 0.000001
    return bigram[(w1, w2)] / unigram[w1]

def entropy_unigram(sentence):
    words = nltk.word_tokenize(sentence.lower())

    probabilities = [
        unigram_prob(word)
        for word in words
    ]

    return -sum(
        p * math.log2(p)
        for p in probabilities
        if p > 0
    ) / len(probabilities)

def entropy_bigram(sentence):
    words = nltk.word_tokenize(sentence.lower())

    probabilities = []

    for i in range(1, len(words)):
        p = bigram_prob(words[i - 1], words[i])
        probabilities.append(p)

    return -sum(
        p * math.log2(p)
        for p in probabilities
        if p > 0
    ) / len(probabilities)

print("=" * 50)
print("ENTROPY-BASED LANGUAGE MODEL")
print("=" * 50)

for sentence in test:

    h_uni = entropy_unigram(sentence)
    h_bi = entropy_bigram(sentence)

    print("\nSentence:")
    print(sentence)

    print(f"Unigram Entropy: {h_uni:.4f}")
    print(f"Bigram Entropy : {h_bi:.4f}")

    if h_bi > 5:
        print("Prediction Uncertainty: HIGH")
    else:
        print("Prediction Uncertainty: LOW")

print("\nINTERPRETATION")
print("Low entropy  = more predictable text")
print("High entropy = less predictable text")
print("Unseen words are assigned a small probability.")

ENTROPY-BASED LANGUAGE MODEL

Sentence:
the cat sits on the mat
Unigram Entropy: 0.3664
Bigram Entropy : 0.2929
Prediction Uncertainty: LOW

Sentence:
the quantum processor redesigned the
Unigram Entropy: 0.2072
Bigram Entropy : 0.0000
Prediction Uncertainty: LOW

INTERPRETATION
Low entropy  = more predictable text
High entropy = less predictable text
Unseen words are assigned a small probability.


[nltk_data] Downloading package punkt to /root/nltk_data...
[nltk_data]   Package punkt is already up-to-date!


In [10]:
#CO3_AT1.4
import nltk
from collections import Counter

nltk.download("punkt")
nltk.download("averaged_perceptron_tagger")

sentences = [
    "I book a ticket.",
    "I read a book."
]


def rule_based(sentence):

    words = nltk.word_tokenize(sentence)

    result = []

    for i, word in enumerate(words):

        if word.lower() == "i":
            tag = "PRP"

        elif word.lower() == "book":

            if i == 1:
                tag = "VB"
            else:
                tag = "NN"

        elif word.lower() in ["a", "an", "the"]:
            tag = "DT"

        elif word.lower() in ["read"]:
            tag = "VB"

        elif word.lower() in ["ticket"]:
            tag = "NN"

        else:
            tag = "NN"

        result.append((word, tag))

    return result




training = [
    ("I", "PRP"),
    ("book", "VB"),
    ("a", "DT"),
    ("ticket", "NN"),
    ("I", "PRP"),
    ("read", "VB"),
    ("a", "DT"),
    ("book", "NN")
]

word_tag_count = Counter(training)

def stochastic(sentence):

    words = nltk.word_tokenize(sentence)
    result = []

    for word in words:

        candidates = [
            (w, tag)
            for (w, tag) in word_tag_count
            if w.lower() == word.lower()
        ]

        if candidates:
            best = max(
                candidates,
                key=lambda x: word_tag_count[x]
            )
            result.append(best)

        else:
            result.append((word, "NN"))

    return result



def transformation_based(sentence):

    words = nltk.word_tokenize(sentence)

    result = []

    for word in words:

        if word.lower() == "i":
            tag = "PRP"

        elif word.lower() in ["a", "an", "the"]:
            tag = "DT"

        elif word.lower() in ["book", "read"]:
            tag = "NN"

        else:
            tag = "NN"

        result.append([word, tag])

    for i in range(len(result)):

        if result[i][0].lower() == "book":

            if i > 0 and result[i - 1][0].lower() == "i":
                result[i][1] = "VB"

    return result




print("COMPARATIVE POS TAGGING SYSTEM")


for sentence in sentences:

    print("\nSENTENCE:")
    print(sentence)

    print("\nRULE-BASED:")
    print(rule_based(sentence))

    print("\nSTOCHASTIC:")
    print(stochastic(sentence))

    print("\nTRANSFORMATION-BASED:")
    print(transformation_based(sentence))

print("\nEXPECTED RESULTS")

print("\nI/PRP book/VB a/DT ticket/NN")
print("I/PRP read/VB a/DT book/NN")

print("\nMETHODS")
print("Rule-Based: uses explicit grammatical rules.")
print("Stochastic: uses word/tag frequencies.")
print("Transformation-Based: starts with initial tags and corrects errors.")

COMPARATIVE POS TAGGING SYSTEM

SENTENCE:
I book a ticket.

RULE-BASED:
[('I', 'PRP'), ('book', 'VB'), ('a', 'DT'), ('ticket', 'NN'), ('.', 'NN')]

STOCHASTIC:
[('I', 'PRP'), ('book', 'VB'), ('a', 'DT'), ('ticket', 'NN'), ('.', 'NN')]

TRANSFORMATION-BASED:
[['I', 'PRP'], ['book', 'VB'], ['a', 'DT'], ['ticket', 'NN'], ['.', 'NN']]

SENTENCE:
I read a book.

RULE-BASED:
[('I', 'PRP'), ('read', 'VB'), ('a', 'DT'), ('book', 'NN'), ('.', 'NN')]

STOCHASTIC:
[('I', 'PRP'), ('read', 'VB'), ('a', 'DT'), ('book', 'VB'), ('.', 'NN')]

TRANSFORMATION-BASED:
[['I', 'PRP'], ['read', 'NN'], ['a', 'DT'], ['book', 'NN'], ['.', 'NN']]

EXPECTED RESULTS

I/PRP book/VB a/DT ticket/NN
I/PRP read/VB a/DT book/NN

METHODS
Rule-Based: uses explicit grammatical rules.
Stochastic: uses word/tag frequencies.
Transformation-Based: starts with initial tags and corrects errors.


[nltk_data] Downloading package punkt to /root/nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package averaged_perceptron_tagger to
[nltk_data]     /root/nltk_data...
[nltk_data]   Unzipping taggers/averaged_perceptron_tagger.zip.


In [4]:
#co3_at2.1
import nltk
import numpy as np
import math
from collections import Counter
from nltk.util import ngrams

corpus = """
machine learning improves business
machine learning enables automation
machine learning drives innovation
""".lower().split()

unigrams = Counter(corpus)
bigrams = Counter(ngrams(corpus, 2))
trigrams = Counter(ngrams(corpus, 3))

total_words = len(corpus)

print("CASE STUDY 1 - E-COMMERCE SMART SEARCH")


print("\nUNIGRAMS")
print(unigrams)

print("\nBIGRAMS")
print(bigrams)

print("\nTRIGRAMS")
print(trigrams)


p_learning_machine = bigrams[("machine", "learning")] / unigrams["machine"]

print("\n1. MLE BIGRAM PROBABILITY")
print("P(learning | machine)")
print(f"= C(machine learning) / C(machine)")
print(f"= 3 / 3")
print(f"= {p_learning_machine:.2f}")


word = "transforms"

if ("learning", word) in bigrams:
    backoff_probability = bigrams[("learning", word)] / unigrams["learning"]
    level = "Bigram"
elif word in unigrams:
    backoff_probability = unigrams[word] / total_words
    level = "Unigram"
else:
    backoff_probability = 1 / total_words
    level = "Unigram fallback"

print("\n2. BACKOFF MODEL")
print("Sequence: machine learning transforms")
print("P(transforms | machine learning)")
print("Trigram unavailable")
print("Bigram unavailable")
print(f"Using {level} probability")
print(f"P(transforms) = 1 / {total_words}")
print(f"Probability = {backoff_probability:.4f}")

lambda1 = 0.5
lambda2 = 0.3
lambda3 = 0.2

p_trigram = 1 / 3
p_bigram = 1 / 3
p_unigram = unigrams["improves"] / total_words

interpolation = (
    lambda1 * p_trigram +
    lambda2 * p_bigram +
    lambda3 * p_unigram
)

print("\n3. DELETED INTERPOLATION")
print(f"Trigram probability = {p_trigram:.4f}")
print(f"Bigram probability  = {p_bigram:.4f}")
print(f"Unigram probability = {p_unigram:.4f}")

print(
    f"P(improves | machine learning) = "
    f"{lambda1}({p_trigram:.4f}) + "
    f"{lambda2}({p_bigram:.4f}) + "
    f"{lambda3}({p_unigram:.4f})"
)

print(f"Final probability = {interpolation:.4f}")


prediction = np.array([0.33, 0.33, 0.33])

entropy = -sum(
    p * math.log2(p)
    for p in prediction
)

print("\n4. ENTROPY")
print("P(improves) = 0.33")
print("P(enables) = 0.33")
print("P(drives) = 0.33")
print(f"Entropy = {entropy:.4f} bits")


next_words = {
    "improves": 0.33,
    "enables": 0.33,
    "drives": 0.33
}

prediction = max(next_words, key=next_words.get)

print("\n5. FINAL NEXT-WORD PREDICTION")
print(f"Candidates: {list(next_words.keys())}")
print(f"Predicted word: {prediction}")

print("\n" + "=" * 60)

CASE STUDY 1 - E-COMMERCE SMART SEARCH

UNIGRAMS
Counter({'machine': 3, 'learning': 3, 'improves': 1, 'business': 1, 'enables': 1, 'automation': 1, 'drives': 1, 'innovation': 1})

BIGRAMS
Counter({('machine', 'learning'): 3, ('learning', 'improves'): 1, ('improves', 'business'): 1, ('business', 'machine'): 1, ('learning', 'enables'): 1, ('enables', 'automation'): 1, ('automation', 'machine'): 1, ('learning', 'drives'): 1, ('drives', 'innovation'): 1})

TRIGRAMS
Counter({('machine', 'learning', 'improves'): 1, ('learning', 'improves', 'business'): 1, ('improves', 'business', 'machine'): 1, ('business', 'machine', 'learning'): 1, ('machine', 'learning', 'enables'): 1, ('learning', 'enables', 'automation'): 1, ('enables', 'automation', 'machine'): 1, ('automation', 'machine', 'learning'): 1, ('machine', 'learning', 'drives'): 1, ('learning', 'drives', 'innovation'): 1})

1. MLE BIGRAM PROBABILITY
P(learning | machine)
= C(machine learning) / C(machine)
= 3 / 3
= 1.00

2. BACKOFF MODEL
Seq

In [5]:
#co3_at2.2
import nltk

nltk.download("punkt", quiet=True)
nltk.download("punkt_tab", quiet=True)
nltk.download("averaged_perceptron_tagger_eng", quiet=True)

sentences = [
    "Book an appointment with the doctor.",
    "The book contains medical information."
]


print("CASE STUDY 2 - HOSPITAL APPOINTMENT CHATBOT")


for sentence in sentences:
    tokens = nltk.word_tokenize(sentence)
    tags = nltk.pos_tag(tokens)

    print("\nSentence:")
    print(sentence)

    print("\nTokens:")
    print(tokens)

    print("\nPenn Treebank POS Tags:")
    print(tags)

print("\nBOOK HMM CALCULATION")

p_book_vb = 0.7
p_book_nn = 0.3

p_start_vb = 0.6
p_start_nn = 0.4

prob_vb = p_start_vb * p_book_vb
prob_nn = p_start_nn * p_book_nn

print("\nP(book | VB) =", p_book_vb)
print("P(Start -> VB) =", p_start_vb)
print("P(VB, book) =", p_start_vb, "*", p_book_vb)
print("=", prob_vb)

print("\nP(book | NN) =", p_book_nn)
print("P(Start -> NN) =", p_start_nn)
print("P(NN, book) =", p_start_nn, "*", p_book_nn)
print("=", prob_nn)

if prob_vb > prob_nn:
    best_tag = "VB"
else:
    best_tag = "NN"

print("\nMOST PROBABLE TAG AT SENTENCE START")
print("Tag:", best_tag)

print("\nCONTEXTUAL TAGGING")

print("Sentence 1:")
print("Book/VB an/DT appointment/NN with/IN the/DT doctor/NN")

print("\nSentence 2:")
print("The/DT book/NN contains/VBZ medical/JJ information/NN")

print("\nRULE-BASED VS HMM")
print("Rule-Based: Uses grammatical rules and context.")
print("HMM: Uses probabilities learned from data.")
print("Rule-Based advantage: Simple and interpretable.")
print("HMM advantage: Handles ambiguity probabilistically.")
print("Limitation: HMM requires reliable training probabilities.")

print("\n" + "=" * 60)

CASE STUDY 2 - HOSPITAL APPOINTMENT CHATBOT

Sentence:
Book an appointment with the doctor.

Tokens:
['Book', 'an', 'appointment', 'with', 'the', 'doctor', '.']

Penn Treebank POS Tags:
[('Book', 'NNP'), ('an', 'DT'), ('appointment', 'NN'), ('with', 'IN'), ('the', 'DT'), ('doctor', 'NN'), ('.', '.')]

Sentence:
The book contains medical information.

Tokens:
['The', 'book', 'contains', 'medical', 'information', '.']

Penn Treebank POS Tags:
[('The', 'DT'), ('book', 'NN'), ('contains', 'VBZ'), ('medical', 'JJ'), ('information', 'NN'), ('.', '.')]

BOOK HMM CALCULATION

P(book | VB) = 0.7
P(Start -> VB) = 0.6
P(VB, book) = 0.6 * 0.7
= 0.42

P(book | NN) = 0.3
P(Start -> NN) = 0.4
P(NN, book) = 0.4 * 0.3
= 0.12

MOST PROBABLE TAG AT SENTENCE START
Tag: VB

CONTEXTUAL TAGGING
Sentence 1:
Book/VB an/DT appointment/NN with/IN the/DT doctor/NN

Sentence 2:
The/DT book/NN contains/VBZ medical/JJ information/NN

RULE-BASED VS HMM
Rule-Based: Uses grammatical rules and context.
HMM: Uses probabi

In [6]:
#co3_at2.3
import nltk
import math
from collections import Counter

nltk.download("punkt", quiet=True)
nltk.download("punkt_tab", quiet=True)
nltk.download("averaged_perceptron_tagger_eng", quiet=True)

sentence = "Market growth drives investment."

initial_tags = [
    ("Market", "NN"),
    ("growth", "NN"),
    ("drives", "NNS"),
    ("investment", "NN")
]

frequencies = {
    "market": 500,
    "growth": 350,
    "drives": 180,
    "investment": 420
}


print("CASE STUDY 3 - FINANCIAL NEWS POS CORRECTION")



tokens = nltk.word_tokenize(sentence)

print("\nTOKENS")
print(tokens)


print("\nINITIAL POS TAGS")
for word, tag in initial_tags:
    print(f"{word}/{tag}")


corrected_tags = initial_tags.copy()

for i in range(1, len(corrected_tags)):
    current_word, current_tag = corrected_tags[i]
    previous_word, previous_tag = corrected_tags[i - 1]

    if current_tag == "NNS" and previous_tag == "NN":
        corrected_tags[i] = (current_word, "VBZ")

print("\nTRANSFORMATION RULE")
print("Change NNS to VBZ if preceding word is NN.")


print("\nCORRECTED POS TAGS")
for word, tag in corrected_tags:
    print(f"{word}/{tag}")


total_frequency = sum(frequencies.values())

probabilities = {}

print("\nWORD FREQUENCY DISTRIBUTION")

for word, frequency in frequencies.items():
    probability = frequency / total_frequency
    probabilities[word] = probability

    print(
        f"{word:12} "
        f"Frequency = {frequency:3} "
        f"Probability = {probability:.4f}"
    )


entropy_before = -sum(
    p * math.log2(p)
    for p in probabilities.values()
)


probabilities_after = probabilities.copy()

entropy_after = -sum(
    p * math.log2(p)
    for p in probabilities_after.values()
)

print("\nENTROPY ANALYSIS")

print(f"Total frequency = {total_frequency}")

print(f"\nEntropy Before Transformation:")
print(f"H = {entropy_before:.4f} bits")

print(f"\nEntropy After Transformation:")
print(f"H = {entropy_after:.4f} bits")

print("\nENTROPY INTERPRETATION")
print("Word frequencies remain unchanged by POS transformation.")
print("Therefore, frequency-distribution entropy remains unchanged.")

print("\nFINAL POS TAGGING RESULT")
print(" ".join(f"{word}/{tag}" for word, tag in corrected_tags))

print("\n" + "=" * 60)

CASE STUDY 3 - FINANCIAL NEWS POS CORRECTION

TOKENS
['Market', 'growth', 'drives', 'investment', '.']

INITIAL POS TAGS
Market/NN
growth/NN
drives/NNS
investment/NN

TRANSFORMATION RULE
Change NNS to VBZ if preceding word is NN.

CORRECTED POS TAGS
Market/NN
growth/NN
drives/VBZ
investment/NN

WORD FREQUENCY DISTRIBUTION
market       Frequency = 500 Probability = 0.3448
growth       Frequency = 350 Probability = 0.2414
drives       Frequency = 180 Probability = 0.1241
investment   Frequency = 420 Probability = 0.2897

ENTROPY ANALYSIS
Total frequency = 1450

Entropy Before Transformation:
H = 1.9161 bits

Entropy After Transformation:
H = 1.9161 bits

ENTROPY INTERPRETATION
Word frequencies remain unchanged by POS transformation.
Therefore, frequency-distribution entropy remains unchanged.

FINAL POS TAGGING RESULT
Market/NN growth/NN drives/VBZ investment/NN

